# RAGAS Benchmark End-to-End (OOP)

Notebook tổng hợp các module mới + module có sẵn để sinh benchmark theo phong cách OOP:
- `GeminiKeyManager` (mới) — xoay vòng key Gemini.
- `EmbeddingService` (có sẵn `src.b_indexing.b1_embedding`) — embedding qua OpenRouter.
- `ChromaVectorDatabase` (có sẵn `src.b_indexing.b0_vector_db`) — load products/policies.
- `ChunkingDocuments` (có sẵn `src.a_ingestion.a4_chunker`) — chunk policies, giữ nguyên products.
- `RAGAS 0.4.3` + `SingleHopSpecificQuerySynthesizer` — sinh câu hỏi.

Cấu trúc OOP trong notebook:
1. `RAGASEmbeddingAdapter`: wrap `EmbeddingService` cho RAGAS, hỗ trợ xoay OpenRouter key.
2. `RAGASBenchmarkGenerator`: pipeline load → chunk → build generator → generate → stats.


In [1]:
import sys
from pathlib import Path

notebook_dir = Path.cwd()
rag_service_dir = str(notebook_dir.parent.resolve())
if rag_service_dir not in sys.path:
    sys.path.append(rag_service_dir)

print("rag-service dir:", rag_service_dir)

rag-service dir: D:\create\Agenttic-RAG-for-e-commerce\rag-service


In [2]:
import asyncio
import time
import json

from configs.GetConfig import config
from configs.setting import settings

# Các module mới
from src.h_evaluation.key_manager import (
    RotatingKeyManager,
    GeminiKeyManager,
    load_keys_from_settings,
)
from src.h_evaluation.ragas_helpers import (
    load_documents,
    to_prechunked_documents,
    build_generator,
    create_single_hop_synthesizer,
    create_multi_hop_synthesizer,
    create_comparison_synthesizer,
    make_ner_transform,
    attach_call_counter,
    format_testset,
)

# Module có sẵn được tái sử dụng
from src.b_indexing.b1_embedding import EmbeddingService

from ragas.run_config import RunConfig

print("Imports OK")
print(f"Gemini keys: {len(load_keys_from_settings(settings, 'GEMINI'))}")
print(f"OpenRouter keys: {len(load_keys_from_settings(settings, 'OPENROUTER'))}")

c:\Users\Admin\anaconda3\envs\DL\Lib\site-packages\instructor\providers\gemini\client.py:6: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Imports OK
Gemini keys: 3
OpenRouter keys: 1


## 1. `RAGASEmbeddingAdapter` — dùng lại `EmbeddingService` + xoay key

In [3]:
class RAGASEmbeddingAdapter:
    """
    Wrap EmbeddingService của dự án để dùng với RAGAS.
    Cung cấp embed_text / aembed_text và tự xoay OpenRouter key khi gặp limit.
    """
    def __init__(self, settings, config, api_keys=None):
        keys = api_keys or [settings.OPENROUTER_API_KEY]
        self.key_manager = RotatingKeyManager(keys)
        self.settings = settings
        self.config = config
        self._service = self._make_service()

    def _make_service(self):
        # Tái sử dụng EmbeddingService đã có sẵn
        return EmbeddingService(
            api_key=self.key_manager.current_key,
            config=self.config,
            settings=self.settings,
        )

    def embed_text(self, text: str):
        last_err = None
        for _ in range(len(self.key_manager) * 2):
            try:
                # max_retries=1 để fail nhanh, rồi xoay key ngoài
                return self._service.get_embedding(text, max_retries=1, timeout=30)
            except ValueError as e:
                last_err = e
                err = str(e).lower()
                if any(k in err for k in ['rate limit', '429', 'quota', 'exhausted']):
                    self.key_manager.rotate()
                    self._service = self._make_service()
                else:
                    raise
        raise last_err

    async def aembed_text(self, text: str):
        loop = asyncio.get_event_loop()
        return await loop.run_in_executor(None, self.embed_text, text)


# Test nhanh
adapter = RAGASEmbeddingAdapter(settings, config)
vec = adapter.embed_text("Laptop MSI giá rẻ")
print("Embedding OK, dim:", len(vec))

Embedding OK, dim: 2048


## 2. `RAGASBenchmarkGenerator` — pipeline OOP end-to-end

In [4]:
class RAGASBenchmarkGenerator:
    """
    Pipeline OOP: load docs từ ChromaDB/CSV, pre-chunk, build RAGAS generator, sinh câu hỏi, trả stats.
    """
    def __init__(
        self,
        settings,
        config,
        gemini_keys=None,
        openrouter_keys=None,
        product_limit=100,
        policy_limit=50,
        query_distribution=None,  # Custom query distribution: [(synth_type, weight), ...]
    ):
        self.settings = settings
        self.config = config
        self.gemini_keys = gemini_keys or load_keys_from_settings(settings, 'GEMINI')
        self.openrouter_keys = openrouter_keys or load_keys_from_settings(settings, 'OPENROUTER')
        self.product_limit = product_limit
        self.policy_limit = policy_limit
        # Default: 40% single-hop, 30% multi-hop, 30% comparison
        self.query_distribution = query_distribution or [
            ('single_hop', 0.4),
            ('multi_hop', 0.3),
            ('comparison', 0.3),
        ]

        # Embedding model dùng lại EmbeddingService
        self.embedding_model = RAGASEmbeddingAdapter(settings, config, self.openrouter_keys)

        self.documents = None
        self.chunks = None
        self.generator = None
        self.llm = None
        self.records = None
        self.stats = None

    def load_documents(self):
        """Load từ ChromaDB, fallback CSV nếu rỗng."""
        self.documents = load_documents(
            limit_products=self.product_limit,
            limit_policies=self.policy_limit,
        )
        self.chunks = to_prechunked_documents(self.documents, self.config)
        print(f"Loaded {len(self.documents)} docs -> {len(self.chunks)} chunks")
        return self

    def build_generator(self):
        """Khởi tạo TestsetGenerator với Gemini LLM."""
        self.gemini_key_manager = GeminiKeyManager(self.gemini_keys)
        self.generator, self.llm = build_generator(
            self.gemini_key_manager,
            self.embedding_model,
        )
        return self

    def generate(self, n_samples=2):
        """Sinh n_samples câu hỏi, trả về records + stats."""
        if self.generator is None:
            self.build_generator()

        counter = {'calls': 0, 'prompt_chars': 0}
        attach_call_counter(self.llm, counter)

        # Build synthesizers based on query_distribution
        synthesizers = []
        for synth_type, weight in self.query_distribution:
            if synth_type == 'single_hop':
                synthesizers.append((create_single_hop_synthesizer(self.llm), weight))
            elif synth_type == 'multi_hop':
                synthesizers.append((create_multi_hop_synthesizer(self.llm), weight))
            elif synth_type == 'comparison':
                synthesizers.append((create_comparison_synthesizer(self.llm), weight))
            else:
                raise ValueError(f"Unknown synthesizer type: {synth_type}")

        ner = make_ner_transform(self.llm)

        t0 = time.time()
        testset = self.generator.generate_with_chunks(
            self.chunks[:n_samples],
            testset_size=n_samples,
            transforms=[ner],
            query_distribution=synthesizers,
            run_config=RunConfig(max_workers=1, max_retries=1, max_wait=20),
            raise_exceptions=True,
        )
        elapsed = time.time() - t0

        self.records = format_testset(testset, prefix='ragas_oop')
        self.stats = {
            'n_questions': len(self.records),
            'llm_calls': counter['calls'],
            'calls_per_question': counter['calls'] / max(len(self.records), 1),
            'elapsed_s': elapsed,
            'time_per_question_s': elapsed / max(len(self.records), 1),
            'prompt_chars': counter['prompt_chars'],
        }
        return self.records, self.stats

    def save(self, output_path=None):
        """Lưu records ra JSONL."""
        if self.records is None:
            raise ValueError("Chưa generate, gọi generate() trước")
        output_path = Path(output_path or rag_service_dir) / 'src/h_evaluation/test_sets/ragas_oop.jsonl'
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with open(output_path, 'w', encoding='utf-8') as f:
            for r in self.records:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print(f"Saved {output_path}")
        return output_path

## 3. Demo: sinh 2 câu hỏi

In [5]:
%%time
# Khởi tạo pipeline
gen = RAGASBenchmarkGenerator(
    settings,
    config,
    product_limit=20,   # giới hạn số products load
    policy_limit=3,     # tất cả policies
)

# Load + generate
gen.load_documents()
records, stats = gen.generate(n_samples=2)

print("\nStats:")
for k, v in stats.items():
    print(f"   {k}: {v}")

print("\nQuestions:")
for i, r in enumerate(records, 1):
    print(f"\n[{i}] Q: {r['question']}")
    print(f"    A: {r['ground_truth'][:200]}...")

   ✅ Loaded 20 products from ChromaDB
   ✅ Loaded 3 policies from ChromaDB
📚 Total documents loaded: 23
Loaded 23 docs -> 23 chunks
CPU times: total: 2.55 s
Wall time: 13.2 s


ImportError: cannot import name 'MultiHopQuerySynthesizer' from 'ragas.testset.synthesizers' (c:\Users\Admin\anaconda3\envs\DL\Lib\site-packages\ragas\testset\synthesizers\__init__.py)

In [6]:
# Lưu kết quả
gen.save()

Saved /home/ubuntu/repos/Agenttic-RAG-for-e-commerce/rag-service/src/h_evaluation/test_sets/ragas_oop.jsonl


PosixPath('/home/ubuntu/repos/Agenttic-RAG-for-e-commerce/rag-service/src/h_evaluation/test_sets/ragas_oop.jsonl')

## 4. Chiến lược 100–200 câu

| Chỉ số | Giá trị ước tính |
|--------|----------------|
| LLM calls / câu | ~2 (1 NER + 1 sample) |
| 200 câu từ ~100 products | ~300 LLM calls |
| Gemini free limit | 500 RPD / 15 RPM |
| Thời gian 1 câu | ~80s (tuần tự) |
| 200 câu 1 key | ~4.5 giờ |
| Nhiều key xoay vòng | giảm tuyến tính |

Câu lệnh cho 200 câu:
```python
gen = RAGASBenchmarkGenerator(settings, config, product_limit=100, policy_limit=50)
gen.load_documents()
records, stats = gen.generate(n_samples=200)
gen.save('src/h_evaluation/test_sets/ragas_200.jsonl')
```
